In [1]:
import os
import torch
import torch.nn as nn
import numpy as np
import json
import pandas as pd

In [2]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: mps


## Load Dataset (tensor file)

In [3]:
data = torch.load("embeddings/torch_tensor_words.pt", map_location="cpu")

In [4]:
project_ids = data["project_ids"]
member_ids  = data["member_ids"]

title_ids   = data["title_ids"]
title_mask  = data["title_mask"]
tag_ids     = data["tag_ids"]
tag_mask    = data["tag_mask"]

member_text_ids  = data["member_text_ids"]
member_text_mask = data["member_text_mask"]
hard_ids   = data["hard_ids"]
hard_mask  = data["hard_mask"]
soft_ids   = data["soft_ids"]
soft_mask  = data["soft_mask"]
int_ids    = data["int_ids"]
int_mask   = data["int_mask"]

## Additive Attention Pooling
learns which tokens matter more and returns a weighted sum

In [5]:
class AdditiveAttentionPool(nn.Module):
    def __init__(self, dim, attn_hidden=128):
        super().__init__()
        self.fc = nn.Linear(dim, attn_hidden)
        self.v  = nn.Linear(attn_hidden, 1, bias=False)

    def forward(self, X, mask):
        scores = self.v(torch.tanh(self.fc(X))).squeeze(-1)
        scores = scores.masked_fill(~mask, -1e9)
        alpha  = torch.softmax(scores, dim=1)
        out    = torch.bmm(alpha.unsqueeze(1), X).squeeze(1)
        return out

In [6]:
class CsdRecStep1Encoder(nn.Module):
    def __init__(
        self,
        word_vocab_size,
        label_vocab_size,
        d_word=128,
        d_label=128,
        n_heads=4,
        dropout=0.1,
        attn_hidden=128,
        use_self_attn_for_member_text=True
    ):
        super().__init__()

        # Trainable lookup tables (this is the key paper idea)
        self.word_emb  = nn.Embedding(word_vocab_size, d_word, padding_idx=0)
        self.label_emb = nn.Embedding(label_vocab_size, d_label, padding_idx=0)

        # Self-attention (paper uses it for title; we also use it for member text optionally)
        self.title_self_attn = nn.MultiheadAttention(
            embed_dim=d_word, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.member_self_attn = nn.MultiheadAttention(
            embed_dim=d_word, num_heads=n_heads, dropout=dropout, batch_first=True
        )

        # Additive attention pooling (paper uses it)
        self.word_pool  = AdditiveAttentionPool(d_word, attn_hidden=attn_hidden)
        self.label_pool = AdditiveAttentionPool(d_label, attn_hidden=attn_hidden)

        self.dropout = nn.Dropout(dropout)
        self.use_self_attn_for_member_text = use_self_attn_for_member_text

    def encode_tasks(self, title_ids, title_mask, tag_ids, tag_mask):
        # ---- Title word stream ----
        X = self.dropout(self.word_emb(title_ids))  # [B, L_title, d_word]

        # key_padding_mask: True means "ignore" (so invert the bool mask)
        key_padding = ~title_mask
        H, _ = self.title_self_attn(X, X, X, key_padding_mask=key_padding)
        H = self.dropout(H)

        c_title = self.word_pool(H, title_mask)  # [B, d_word]

        # ---- Tag label stream ----
        Z = self.dropout(self.label_emb(tag_ids))  # [B, L_tags, d_label]
        c_tags = self.label_pool(Z, tag_mask)      # [B, d_label]

        # Final task text embedding
        c_v = torch.cat([c_title, c_tags], dim=1)  # [B, d_word + d_label]
        return c_v

    def encode_members(self, member_text_ids, member_text_mask,
                       hard_ids, hard_mask, soft_ids, soft_mask, int_ids, int_mask):

        # ---- Member text stream (bio + descriptions) ----
        X = self.dropout(self.word_emb(member_text_ids))  # [B, L_text, d_word]

        if self.use_self_attn_for_member_text:
            key_padding = ~member_text_mask
            H, _ = self.member_self_attn(X, X, X, key_padding_mask=key_padding)
            H = self.dropout(H)
            c_text = self.word_pool(H, member_text_mask)
        else:
            # If you want simpler: no self-attn, just pool the word embeddings directly
            c_text = self.word_pool(X, member_text_mask)

        # ---- Member label streams ----
        Zh = self.dropout(self.label_emb(hard_ids))
        Zs = self.dropout(self.label_emb(soft_ids))
        Zi = self.dropout(self.label_emb(int_ids))

        c_hard = self.label_pool(Zh, hard_mask)
        c_soft = self.label_pool(Zs, soft_mask)
        c_int  = self.label_pool(Zi, int_mask)

        # Final member embedding
        c_u = torch.cat([c_text, c_hard, c_soft, c_int], dim=1)  # [B, d_word + 3*d_label]
        return c_u


## Initialize the model

In [7]:
with open("embeddings/word2id.json", "r") as f:
    word2id = json.load(f)

with open("embeddings/label2id.json", "r") as f:
    label2id = json.load(f)

In [8]:
word_vocab_size  = len(word2id)
label_vocab_size = len(label2id)

In [9]:
model = CsdRecStep1Encoder(
    word_vocab_size=word_vocab_size,
    label_vocab_size=label_vocab_size,
    d_word=128,
    d_label=128,
    n_heads=4,
    dropout=0.1,
    attn_hidden=128,
    use_self_attn_for_member_text=True  # keep True (good)
).to(device)

## Run embeddings in batches

In [10]:
def batched_encode_tasks(batch_size=1024):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, title_ids.size(0), batch_size):
            c_v = model.encode_tasks(
                title_ids[i:i+batch_size].to(device),
                title_mask[i:i+batch_size].to(device),
                tag_ids[i:i+batch_size].to(device),
                tag_mask[i:i+batch_size].to(device),
            )
            outs.append(c_v.cpu())
    return torch.cat(outs, dim=0)

In [11]:
def batched_encode_members(batch_size=1024):
    model.eval()
    outs = []
    with torch.no_grad():
        for i in range(0, member_text_ids.size(0), batch_size):
            c_u = model.encode_members(
                member_text_ids[i:i+batch_size].to(device),
                member_text_mask[i:i+batch_size].to(device),
                hard_ids[i:i+batch_size].to(device),
                hard_mask[i:i+batch_size].to(device),
                soft_ids[i:i+batch_size].to(device),
                soft_mask[i:i+batch_size].to(device),
                int_ids[i:i+batch_size].to(device),
                int_mask[i:i+batch_size].to(device),
            )
            outs.append(c_u.cpu())
    return torch.cat(outs, dim=0)

In [12]:
c_projects = batched_encode_tasks(batch_size=512)
c_members = batched_encode_members(batch_size=512)

print("c_projects shape:", c_projects.shape)  # [num_projects, d_word + d_label]
print("c_members shape:", c_members.shape)  # [num_members, d_word + 3*d_label]

c_projects shape: torch.Size([18569, 256])
c_members shape: torch.Size([38462, 512])


## Save embeddings

In [13]:
np.save("embeddings/project_initial_embeddings.npy", c_projects.numpy())
np.save("embeddings/member_initial_embeddings.npy", c_members.numpy())

pd.DataFrame({"project_id": project_ids}).to_csv("embeddings/project_ids.csv", index=False)
pd.DataFrame({"member_id": member_ids}).to_csv("embeddings/member_ids.csv", index=False)